In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import col

# 1. Initialize Spark with Balanced Resources
spark = (
    SparkSession.builder
    .appName("Business_Risk_Analysis")
    # Memory & Performance
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.instances", "1")
    .config("spark.executor.cores", "2")
    .config("spark.driver.maxResultSize", "1g")
    # NLP & Off-Heap Memory
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "2g")
    # Serialization & Networking
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryoserializer.buffer.max", "2000M")
    .config("spark.network.timeout", "1200s")
    # SQL Optimizations
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [2]:
# 2. Load the Full Dataset
risk_df = spark.read.parquet("hdfs://namenode:8020/google_local/Risk_Aggregated")

In [3]:
# 3. Define Pipeline Components
label_indexer = StringIndexer(inputCol="risk_label", outputCol="label")

In [4]:
feature_cols = [
    "total_reviews", "avg_user_rating", "avg_sentiment_score",
    "negative_review_ratio", "yearly_negative_trend", "rating_sentiment_gap"
]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

rf = RandomForestClassifier(
    labelCol="label", 
    featuresCol="features", 
    numTrees=100, 
    maxDepth=7, 
    seed=42
)

pipeline = Pipeline(stages=[label_indexer, assembler, rf])

In [5]:
# 4. Create the Training Sample (25% for stability)
# This prevents the "Connection Refused" error while keeping patterns accurate
sampled_df = risk_df.sample(withReplacement=False, fraction=0.30, seed=42)
train_df, test_df = sampled_df.randomSplit([0.7, 0.3], seed=42)
train_df.cache()

DataFrame[gmap_id: string, name: string, state: string, year: int, total_reviews: bigint, avg_user_rating: double, avg_sentiment_score: double, negative_review_ratio: double, prev_neg_ratio: double, yearly_negative_trend: double, rating_sentiment_gap: double, risk_score: double, risk_label: string]

In [6]:
# 5. Train the Model
print("Starting Model Training...")
rf_model = pipeline.fit(train_df)
print("Training Complete!")

Starting Model Training...
Training Complete!


In [7]:
# 6. Evaluation (On the 30% Test Subset)
test_predictions = rf_model.transform(test_df)
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

accuracy = evaluator.evaluate(test_predictions, {evaluator.metricName: "accuracy"})
f1 = evaluator.evaluate(test_predictions, {evaluator.metricName: "f1"})
print(f"Validation Results -> Accuracy: {accuracy:.4f}, F1-Score: {f1:.4f}")

Validation Results -> Accuracy: 0.9972, F1-Score: 0.9972


In [8]:
# 1. Generate predictions for the Training data
train_predictions = rf_model.transform(train_df)

In [9]:
# 2. Setup Evaluator
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

In [10]:
# 3. Calculate Training Accuracy
train_accuracy = evaluator.evaluate(train_predictions)

In [11]:
# 4. Calculate Testing Accuracy (Already calculated in your script, but let's re-print it)
test_accuracy = evaluator.evaluate(test_predictions)

In [12]:
print("-" * 30)
print(f"TRAINING ACCURACY: {train_accuracy:.4f}")
print(f"TESTING ACCURACY:  {test_accuracy:.4f}")
print("-" * 30)

------------------------------
TRAINING ACCURACY: 0.9972
TESTING ACCURACY:  0.9972
------------------------------


In [13]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Precision
precision = evaluator.evaluate(
    test_predictions, 
    {evaluator.metricName: "weightedPrecision"}
)

# Recall
recall = evaluator.evaluate(
    test_predictions, 
    {evaluator.metricName: "weightedRecall"}
)

# F1 Score
f1_score = evaluator.evaluate(
    test_predictions, 
    {evaluator.metricName: "f1"}
)

print("\n" + "="*40)
print(f"PRECISION (Weighted): {precision:.4f}")
print(f"RECALL (Weighted):    {recall:.4f}")
print(f"F1-SCORE (Weighted):  {f1_score:.4f}")
print("="*40)


PRECISION (Weighted): 0.9972
RECALL (Weighted):    0.9972
F1-SCORE (Weighted):  0.9972


In [14]:
from pyspark.sql.functions import col

confusion_matrix = test_predictions.groupBy("label", "prediction") \
    .count() \
    .orderBy("label", "prediction")

confusion_matrix.show()

+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|  0.0|       0.0|1852001|
|  0.0|       1.0|   3139|
|  1.0|       0.0|   2210|
|  1.0|       1.0| 187031|
|  1.0|       2.0|    142|
|  2.0|       1.0|    511|
|  2.0|       2.0|  80284|
+-----+----------+-------+



In [13]:
print("Generating final predictions for 100% of the dataset...")
final_results = rf_model.transform(risk_df)

Generating final predictions for 100% of the dataset...


In [14]:
from pyspark.sql.functions import first

# Load metadata and remove duplicates BEFORE join
category_lookup = (
    spark.read.json("hdfs://namenode:8020/google_local/total_metadata_raw")
    .select("gmap_id", "category")
    .groupBy("gmap_id")
    .agg(first("category").alias("category"))
)

# Now join safely
results_with_category = final_results.join(
    category_lookup,
    on="gmap_id",
    how="left"
)


In [15]:
results_with_category.printSchema()

root
 |-- gmap_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- total_reviews: long (nullable = true)
 |-- avg_user_rating: double (nullable = true)
 |-- avg_sentiment_score: double (nullable = true)
 |-- negative_review_ratio: double (nullable = true)
 |-- prev_neg_ratio: double (nullable = true)
 |-- yearly_negative_trend: double (nullable = true)
 |-- rating_sentiment_gap: double (nullable = true)
 |-- risk_score: double (nullable = true)
 |-- risk_label: string (nullable = true)
 |-- label: double (nullable = false)
 |-- features: vector (nullable = true)
 |-- rawPrediction: vector (nullable = true)
 |-- probability: vector (nullable = true)
 |-- prediction: double (nullable = false)
 |-- category: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [17]:
def output_exists(path):
    try:
        # Check if directory exists and has data
        return spark.read.parquet(path).limit(1).count() > 0
    except:
        return False

In [24]:
states = [
    "California","Colorado",
    "Connecticut","Delaware","District_of_Columbia","Florida","Georgia","Hawaii",
    "Idaho","Illinois","Indiana","Iowa","Kansas","Kentucky",
    "Louisiana","Maine","Maryland","Massachusetts","Michigan","Minnesota",
    "Mississippi","Missouri","Montana","Nebraska","Nevada","New_Hampshire",
    "New_Jersey","New_Mexico","New_York","North_Carolina","North_Dakota","Ohio",
    "Oklahoma","Oregon","Pennsylvania","Rhode_Island","South_Carolina","South_Dakota",
    "Tennessee","Texas","Utah","Vermont","Virginia","Washington",
    "West_Virginia","Wisconsin","Wyoming"
]

In [25]:
from pyspark.sql.functions import col

# Define your base output path for the frontend
base_output_path = "hdfs://namenode:8020/google_local/Updated_Risk_Frontend_Data"

for state in states:
    print(f"\n{'='*60}")
    print(f"Processing STATE: {state}")
    print(f"{'='*60}")

    try:
        # Step A: Filter all years for this SPECIFIC state from your ML results
        # We select the columns needed for your 5 Use Cases here
        state_data = results_with_category.filter(col("state") == state).select(
            "gmap_id", "name", "state", "year", "category",
            "total_reviews", "avg_user_rating", "avg_sentiment_score",
            "negative_review_ratio", "yearly_negative_trend", "rating_sentiment_gap",
            "risk_score", "risk_label", "probability", "prediction"
        ).cache() # Cache so the year-loop below is lightning fast

        # Step B: Identify which years exist for this state
        years = [row.year for row in state_data.select("year").distinct().orderBy("year").collect()]
        print(f"Years found in {state}: {years}")

        for y in years:
            # We use the standard HDFS partition naming: state=Name/year=YYYY
            # This is critical for the Frontend's performance
            output_path = f"{base_output_path}/state={state}/year={y}"

            # Step C: Checkpoint - skip if already done
            if output_exists(output_path):
                print(f"  Year {y} already exists → skipping")
                continue

            print(f"  Saving year: {y}...", end="\r")
            
            # Step D: Save the partition
            (
                state_data.filter(col("year") == y)
                .repartition(1) # Keeps HDFS clean for small year-files
                .write
                .mode("overwrite")
                .parquet(output_path)
            )
            print(f"  Year {y} saved successfully.")

        # Step E: CRITICAL - Clear RAM for the next state
        state_data.unpersist()
        print(f"Finished STATE: {state}")

    except Exception as e:
        print(f" Failed state {state}: {str(e)[:150]}")
        continue




Processing STATE: California
Years found in California: [1990, 1999, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
  Year 1990 saved successfully.
  Year 1999 saved successfully.
  Year 2002 saved successfully.
  Year 2003 saved successfully.
  Year 2004 saved successfully.
  Year 2005 saved successfully.
  Year 2006 saved successfully.
  Year 2007 saved successfully.
  Year 2008 saved successfully.
  Year 2009 saved successfully.
  Year 2010 saved successfully.
  Year 2011 saved successfully.
  Year 2012 saved successfully.
  Year 2013 saved successfully.
  Year 2014 saved successfully.
  Year 2015 saved successfully.
  Year 2016 saved successfully.
  Year 2017 saved successfully.
  Year 2018 saved successfully.
  Year 2019 saved successfully.
  Year 2020 saved successfully.
  Year 2021 saved successfully.
Finished STATE: California

Processing STATE: Colorado
Years found in Colorado: [1990, 2006, 2007, 2008, 2

In [20]:
alaskatest2_df = spark.read.parquet(
    "hdfs://namenode:8020/google_local/Updated_Risk_Frontend_Data/state=Alaska"
)

alaskatest2_df.printSchema()

root
 |-- gmap_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- category: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- total_reviews: long (nullable = true)
 |-- avg_user_rating: double (nullable = true)
 |-- avg_sentiment_score: double (nullable = true)
 |-- negative_review_ratio: double (nullable = true)
 |-- yearly_negative_trend: double (nullable = true)
 |-- rating_sentiment_gap: double (nullable = true)
 |-- risk_score: double (nullable = true)
 |-- risk_label: string (nullable = true)
 |-- probability: vector (nullable = true)
 |-- prediction: double (nullable = true)



In [21]:
from pyspark.sql.functions import col

# Filter for the name, select specific columns, and sort by year
zen_ratios = (
    alaskatest2_df
    .filter(col("name").contains("Fisherman's Bend"))
    .select("gmap_id","state","category","name", "year", "risk_label", "prediction","probability")
    .orderBy("year")
)

zen_ratios.show()

+--------------------+------+--------------------+----------------+----+----------+----------+--------------------+
|             gmap_id| state|            category|            name|year|risk_label|prediction|         probability|
+--------------------+------+--------------------+----------------+----+----------+----------+--------------------+
|0x54011df24585552...|Alaska|[Gas station, Mar...|Fisherman's Bend|2016|       LOW|       0.0|[0.99991710152712...|
|0x54011df24585552...|Alaska|[Gas station, Mar...|Fisherman's Bend|2018|       LOW|       0.0|[0.99898940846605...|
|0x54011df24585552...|Alaska|[Gas station, Mar...|Fisherman's Bend|2019|       LOW|       0.0|[0.99987703439978...|
|0x54011df24585552...|Alaska|[Gas station, Mar...|Fisherman's Bend|2020|      HIGH|       2.0|[0.0,0.0405079122...|
|0x54011df24585552...|Alaska|[Gas station, Mar...|Fisherman's Bend|2021|       LOW|       0.0|[0.99991710152712...|
+--------------------+------+--------------------+----------------+----+

In [2]:
alaskatest_df = spark.read.parquet(
    "hdfs://namenode:8020/google_local/Risk_Frontend_Data/state=Alaska"
)

alaskatest_df.printSchema()


root
 |-- gmap_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- category: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- total_reviews: long (nullable = true)
 |-- avg_user_rating: double (nullable = true)
 |-- avg_sentiment_score: double (nullable = true)
 |-- negative_review_ratio: double (nullable = true)
 |-- yearly_negative_trend: double (nullable = true)
 |-- rating_sentiment_gap: double (nullable = true)
 |-- risk_score: double (nullable = true)
 |-- risk_label: string (nullable = true)
 |-- probability: vector (nullable = true)
 |-- prediction: double (nullable = true)



In [4]:
# Displays one row in a vertical format so you can read every column clearly
alaskatest_df.show(n=5, vertical=True, truncate=False)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------
 gmap_id               | 0x50c2d807b326b559:0x65a34e0ff0b63f43                                                                                                              
 name                  | GCI Store | Utqiagvik                                                                                                                              
 state                 | Alaska                                                                                                                                             
 year                  | 2019                                                                                                                                               
 category              | [Telecommunications service provider, Business to business service, Cable company, Cell phone store, Internet 

In [38]:
from pyspark.sql.functions import col

# Filter for the name, select specific columns, and sort by year
zen_ratios = (
    alaskatest_df
    .filter(col("name").contains("Dr. David M. Witham, MD"))
    .select("state","category","name", "year", "risk_label", "prediction","probability")
    .orderBy("year")
)

zen_ratios.show()

+------+--------------------+--------------------+----+----------+----------+--------------------+
| state|            category|                name|year|risk_label|prediction|         probability|
+------+--------------------+--------------------+----+----------+----------+--------------------+
|Alaska|[Orthopedic surgeon]|Dr. David M. With...|2018|       LOW|       0.0|[0.99987703439978...|
|Alaska|[Orthopedic surgeon]|Dr. David M. With...|2019|       LOW|       0.0|[0.99991710152712...|
|Alaska|[Orthopedic surgeon]|Dr. David M. With...|2021|      HIGH|       2.0|[0.0,0.0405079122...|
+------+--------------------+--------------------+----+----------+----------+--------------------+

